In [ ]:
import time

import numpy as np
from torch.utils.data import DataLoader
import pandas as pd

from sklearn import metrics
from sklearn.decomposition import PCA
from collections import deque
from sklearn.manifold import TSNE

import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import datasets, transforms

In [ ]:
# ==========================================================================
# 1. Data load
# ==========================================================================

def load_data():
    Fashion_mnist_test_transform = transforms.Compose(
        [transforms.ToTensor()]
    )

    testset_Fashion_mnist = datasets.FashionMNIST(
        root='./data',
        train=False,
        download=True,
        transform=Fashion_mnist_test_transform
    )

    FM_test = DataLoader(
        testset_Fashion_mnist,
        batch_size=32,
        shuffle=False,
        num_workers=2
    )

    FM_test_images = []
    FM_test_labels = []

    for batch in FM_test:
        images, labels=batch
        images_flat=images.view(images.shape[0], -1)
        FM_test_images.append(images_flat.numpy())
        FM_test_labels.append(labels.numpy())
    FM_test_images=np.vstack(FM_test_images)
    FM_test_labels=np.concatenate(FM_test_labels)

    X_=pd.DataFrame(data=FM_test_images)
    y_=pd.Series(data=FM_test_labels)

    print("Data loaded successfully.")
    print("X shape:", X_.shape)
    print("y shape:", y_.shape)

    return X_, y_

In [ ]:
# ==========================================================================
# 2. PCA
# ==========================================================================

def principal_component_analysis(X_, y_, n_components):
    pca = PCA(n_components)
    test_PCA = pca.fit_transform(X_)
    test_PCA = pd.DataFrame(data=test_PCA)
    
    testDF=pd.DataFrame(data=test_PCA.loc[:, 0:1], index=test_PCA.index)
    testDF=pd.concat((testDF, y_), axis=1, join="inner")
    testDF.columns=["x-axis", "y-axis", "Label"]
    sns.lmplot(
        x="x-axis",
        y="y-axis",
        hue="Label",
        data=testDF,
        fit_reg=False,
        height=8
    )
    plt.grid()

    plt.show()

    return test_PCA

In [ ]:
# ==========================================================================
# 3. ARI
# ==========================================================================

def adjusted_rand_index(y_, cluster_labels):
    ari = metrics.adjusted_rand_score(y_, cluster_labels)

    return ari

In [ ]:
# ==========================================================================
# 4. K-means clustering
# ==========================================================================

def k_means_pp(X_, n_clusters, random_state = 42):
    X = np.asarray(X_, dtype = np.float32)
    n_samples = X.shape[0]

    rng = np.random.default_rng(random_state)

    centroids = np.empty((n_clusters, X.shape[1]), dtype = np.float32)

    first_idx = rng.integers(n_samples)
    centroids[0] = X[first_idx]

    # squared dist
    closest_dist = np.sum((X - centroids[0]) ** 2, axis = 1)

    for c in range(1, n_clusters):
        # total dist
        total_dist = np.sum(closest_dist)

        if total_dist == 0:
            next_idx = rng.integers(n_samples)
        else:
            probabilities = closest_dist / total_dist
            next_idx = rng.choice(n_samples, p = probabilities)

        centroids[c] = X[next_idx]

        new_dist = np.sum((X - centroids[c]) ** 2, axis = 1)
        closest_dist = np.minimum(closest_dist, new_dist)

    return centroids

def k_means_clustering(
    X_,
    clusters=10,
    max_iter=100,
    n_init=5,
    tol=1e-4,
    random_state=42
):

    X = np.asarray(X_, dtype=np.float32)
    n_samples = X.shape[0]

    best_labels = None
    best_inertia = np.inf

    for init_idx in range(n_init):
        centroids = k_means_pp(X, clusters, random_state + init_idx)

        for iteration in range(max_iter):
            distances = np.sum((X[:, np.newaxis] - centroids) ** 2, axis=2)

            labels = np.argmin(distances, axis=1)

            new_centroids = np.empty_like(centroids)

            for c in range(clusters):
                cluster_points = X[labels == c]

                if len(cluster_points) == 0:
                    farthest_idx = np.argmax(np.min(distances, axis=1))
                    new_centroids[c] = X[farthest_idx]
                else:
                    new_centroids[c] = np.mean(cluster_points, axis=0)

            centroids_shift = np.sqrt(np.sum((centroids - new_centroids) ** 2))

            centroids = new_centroids

            if centroids_shift < tol:
                break
        
        final_distances = np.sum((X[:, np.newaxis] - centroids) ** 2, axis=2)

        final_labels = np.argmin(final_distances, axis=1)
        inertia = np.sum(np.min(final_distances, axis=1))

        if inertia < best_inertia:
            best_inertia = inertia
            best_labels = final_labels

    return best_labels


In [ ]:
# ==========================================================================
# 5. DBSCAN clustering
# ==========================================================================

def dbscan_clustering(X_, eps, min_neighbors):
    X = np.asarray(X_, dtype=np.float32)
    n_samples = X.shape[0]

    cluster_labels = np.full(n_samples, -np.inf, dtype = int)

    cluster_id = 0

    def find_neighbors(point_idx):
        diff = X - X[point_idx]
        distances = np.sqrt(np.sum(diff * diff, axis=1))

        neighbors = np.where(distances <= eps)[0]

        return neighbors

    for i in range(n_samples):
        if cluster_labels[i] != -np.inf:
            continue

        neighbors = find_neighbors(i)

        if len(neighbors) < min_neighbors:
            cluster_labels[i] = -1
            continue

        cluster_labels[i] = cluster_id

        queue = deque(neighbors)

        while queue:
            neighbor_idx = queue.popleft()

            if cluster_labels[neighbor_idx] == -1:
                cluster_labels[neighbor_idx] = cluster_id

            if cluster_labels[neighbor_idx] != -np.inf:
                continue

            cluster_labels[neighbor_idx] = cluster_id

            neighbor_neighbors = find_neighbors(neighbor_idx)

            if len(neighbor_neighbors) >= min_neighbors:
                queue.extend(neighbor_neighbors)

        cluster_id += 1

    return cluster_labels

def dbscan_grid_search(X_, y_, dim, eps_list, min_neighbor_list):

    best_ari = 0
    best_result = None
    best_labels = None
    grid_results = []

    for eps in eps_list:
        for min_neighbors in min_neighbor_list:
            print()
            print("-" * 60)
            print(f"DBSCAN trial | dim={dim}, eps={eps}, min_neighbors={min_neighbors}")

            start_time = time.time()

            cluster_labels = dbscan_clustering(
                X_,
                eps=eps,
                min_neighbors=min_neighbors
            )

            end_time = time.time()
            elapsed = end_time - start_time

            ari = adjusted_rand_index(y_, cluster_labels)

            n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
            n_noise = np.sum(cluster_labels == -1)
            noise_ratio = n_noise / len(cluster_labels)

            result = {
                "Algorithm": "DBSCAN",
                "Dim": dim,
                "ARI": ari,
                "Time": elapsed,
                "eps": eps,
                "min_neighbors": min_neighbors,
                "n_clusters": n_clusters
            }

            grid_results.append(result)

            print(
                f"ARI={ari:.4f} | clusters={n_clusters} | "
                f"noise={n_noise} ({noise_ratio:.3f}) | time={elapsed:.2f}s"
            )
            print("-" * 60)

            if n_clusters > 1 and ari > best_ari:
                best_ari = ari
                best_result = result
                best_labels = cluster_labels.copy()

    print()
    print("-" * 60)
    print(f"Best result | dim={dim} | eps={best_result['eps']} | min_neighbors={best_result['min_neighbors']}")
    print(f"ARI={best_result['ARI']:.4f} | clusters={best_result['n_clusters']} | time={best_result['Time']:.2f}s")

    return best_result, best_labels, grid_results


In [ ]:
# ==========================================================================
# 6. Sampling
# ==========================================================================

def sample_for_visualization(X_, labels_, sample_size):
    X = np.asarray(X_, dtype=np.float32)
    labels = np.asarray(labels_)

    n_samples = X.shape[0]
    sample_size = min(sample_size, n_samples)

    random_state = 42
    rng = np.random.default_rng(random_state)
    sampled_indices = rng.choice(n_samples, size=sample_size, replace=False)

    X_sample = X[sampled_indices]
    label_sample = labels[sampled_indices]

    return X_sample, label_sample

In [ ]:
# ==========================================================================
# 7. t-SNE
# ==========================================================================

def t_stochastic_neighbor_embedding(X_, best_labels, sample_size=100):
    X_sample, label_sample = sample_for_visualization(
        X_,
        best_labels,
        sample_size=sample_size
    )

    n_components=2
    learning_rate=300
    perplexity=30
    early_exaggeration=12
    init="random"

    tSNE=TSNE(
        n_components=n_components,
        learning_rate=learning_rate,
        perplexity=perplexity,
        early_exaggeration=early_exaggeration,
        init=init
    )

    X_test_tSNE=tSNE.fit_transform(X_sample)
    X_test_tSNE=pd.DataFrame(data=X_test_tSNE)
    testDF=pd.DataFrame(data=X_test_tSNE.loc[:, :], index=X_test_tSNE.index)
    testDF=pd.concat((testDF, pd.Series(label_sample)), axis=1, join="inner")
    testDF.columns=["x-axis", "y-axis", "Label"]

    sns.lmplot(
        x="x-axis",
        y="y-axis",
        hue="Label",
        data=testDF,
        fit_reg=False,
        height=8
    )
    plt.title("Clustering Result")
    plt.grid()

    plt.show()

    return

In [ ]:
# ==========================================================================
# 8. Experiment
# ==========================================================================

def experiment():
    # sample_size
    sample_size = 1000

    results = []
    dbscan_grid_results = []

    # load data
    X_, y_ = load_data()

    # PCA
    X_784 = X_
    X_100 = principal_component_analysis(X_, y_, n_components=100)
    X_50 = principal_component_analysis(X_, y_, n_components=50)
    X_10 = principal_component_analysis(X_, y_, n_components=10)

    datasets = {
        784: X_784,
        100: X_100,
        50: X_50,
        10: X_10
    }

    # k-means
    for dim, X_dim in datasets.items():
        print()
        print("=" * 74)
        print("K-means clustering")

        start_time = time.time()

        cluster_labels = k_means_clustering(
            X_dim,
            clusters=10,
            max_iter=100,
            n_init=5,
            tol=1e-4,
            random_state=42
        )

        end_time = time.time()
        elapsed = end_time - start_time

        ari = adjusted_rand_index(y_, cluster_labels)

        print(f"dimension={dim} | ARI={ari:.4f} | time={elapsed:.2f}s")
        print("=" * 74)

        # t_SNE
        t_stochastic_neighbor_embedding(
            X_dim,
            cluster_labels,
            sample_size=sample_size
        )

        results.append({
            "Algorithm": "K-means",
            "Dim": dim,
            "ARI": round(ari, 4),
            "Time": round(elapsed, 2),
            "Parameters": "k=10, k-means++, n_init=5",
            "n_clusters": 10
        })

    # DBSCAN
    for dim, X_dim in datasets.items():
        print()
        print("=" * 74)
        print("DBSCAN clustering")

        # 차원별 하이퍼 파라미터
        if dim == 784:
            eps_list = [2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
            min_neighbor_list = [10, 15, 20]

        elif dim == 100:
            eps_list = [2.0, 2.5, 3.0, 3.5, 4.0]
            min_neighbor_list = [5, 10, 15, 20]

        elif dim == 50:
            eps_list = [1.5, 2.0, 2.5, 3.0, 3.5]
            min_neighbor_list = [5, 10, 15, 20]

        elif dim == 10:
            eps_list = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
            min_neighbor_list = [5, 10, 15]

        best_result, best_labels, grid_results = dbscan_grid_search(
            X_=X_dim,
            y_=y_,
            dim=dim,
            eps_list=eps_list,
            min_neighbor_list=min_neighbor_list
        )

        dbscan_grid_results.extend(grid_results)

        print("=" * 74)

        # t_SNE
        t_stochastic_neighbor_embedding(
            X_dim,
            best_labels,
            sample_size=sample_size
        )

        results.append({
            "Algorithm": "DBSCAN",
            "Dim": dim,
            "ARI": round(best_result["ARI"], 4),
            "Time": round(best_result["Time"], 2),
            "Parameters": (
                f"eps={best_result['eps']}, "
                f"min_neighbors={best_result['min_neighbors']}"
            ),
            "n_clusters": best_result["n_clusters"]
        })

    # results
    result_df = pd.DataFrame(results)
    dbscan_df = pd.DataFrame(dbscan_grid_results)

    print()
    print("=" * 74)
    print("Final Results")
    print("-" * 74)
    print(result_df)
    print("=" * 74)

    print()
    print("=" * 74)
    print("DBSCAN Grid Search Results")
    print("-" * 74)
    print(dbscan_df)
    print("=" * 74)

    return

In [ ]:
# ==========================================================================
# 9. Main
# ==========================================================================

def main():
    experiment()
    return

main()